# ID10M GPT-4o Zero-Shot Eval
Compares GPT-4o zero-shot vs System E (mBERT) zero-shot on ID10M.  
Metric: CLS macro F1 (idiomatic/literal). No span extraction.

**Prerequisites:** Download from Drive → `IdiomatorRigor/id10m_eval/`:
- `id10m_system_e_english_preds.jsonl`
- `id10m_system_e_spanish_preds.jsonl`
- `id10m_results.json`

**Cost:** ~$0.25 for EN+ES.

In [1]:
%pip install openai scikit-learn tqdm ipywidgets -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, time
from pathlib import Path
from openai import OpenAI
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
from dotenv import load_dotenv
load_dotenv()
# ── CONFIG ────────────────────────────────────────────────────────────────────
API_KEY    = os.getenv('OPENAI_API_KEY')   # or paste key here
TSV_DIR    = Path('../ID10M_test_eval')              # flat TSVs: test_english.tsv etc.
OUTPUT_DIR = Path('ID10M_test_eval_Output')          # saves gpt4o preds + results here
MBERT_JSON = None                                      # optional: Path to id10m_results.json from Drive
LANGS      = ['EN', 'ES']
MODEL      = 'gpt-4o'
DRY_RUN    = None    # set to int (e.g. 5) to test cheaply
SLEEP      = 0.3
FORCE      = False

client = OpenAI(api_key=API_KEY)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config OK')

Config OK


In [4]:
# ── LOAD EXAMPLES (from flat TSV files) ──────────────────────────────────────
TSV_MAP = {
    'EN': ('English', 'test_english.tsv'),
    'ES': ('Spanish', 'test_spanish.tsv'),
}
PUNCT = set('.,;:!?)]\'\"…—–')

def reconstruct_sentence(tokens):
    s = ''
    for tok in tokens:
        if s and tok not in PUNCT and not tok.startswith("'"):
            s += ' '
        s += tok
    return s

def parse_bio_tsv(tsv_path, lang_label):
    examples, tokens, tags, sent_id = [], [], [], 0

    def flush(tokens, tags, sid):
        if not tokens: return None
        return {
            'id10m_id':     f'{lang_label}_{sid}',
            'language':     lang_label,
            'idiomaticity': 'idiomatic' if any(t == 'B-IDIOM' for t in tags) else 'literal',
            'sentence':     reconstruct_sentence(tokens),
        }

    with open(tsv_path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if not line.strip():
                ex = flush(tokens, tags, sent_id)
                if ex: examples.append(ex); sent_id += 1
                tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    tokens.append(parts[0].strip())
                    tags.append(parts[1].strip())
    ex = flush(tokens, tags, sent_id)
    if ex: examples.append(ex)
    return examples

lang_examples = {}
for code in LANGS:
    if code not in TSV_MAP:
        print(f'[{code}] No TSV mapping defined')
        continue
    label, fname = TSV_MAP[code]
    tsv_path = TSV_DIR / fname
    if not tsv_path.exists():
        print(f'[{code}] NOT FOUND: {tsv_path}')
        continue
    examples = parse_bio_tsv(tsv_path, label)
    lang_examples[code] = examples
    dist = {k: sum(1 for e in examples if e['idiomaticity']==k) for k in ('idiomatic','literal')}
    print(f'[{code}] {len(examples)} sentences | {dist}')

# mBERT comparison (optional)
mbert_results = {}
if MBERT_JSON and Path(MBERT_JSON).exists():
    mbert_results = json.loads(Path(MBERT_JSON).read_text())
    print('\nmBERT results loaded:')
    for lang, m in mbert_results.items():
        print(f'  [{lang}] macro_f1={m["macro_f1"]:.4f}')
else:
    print('\nmBERT results not loaded — skipping comparison column')

[EN] 200 sentences | {'idiomatic': 159, 'literal': 41}
[ES] 199 sentences | {'idiomatic': 133, 'literal': 66}

mBERT results not loaded — skipping comparison column


In [5]:
# ── GPT-4o CLASSIFICATION ─────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a linguist specializing in idiomatic language.

Given a sentence, classify whether it uses language IDIOMATICALLY (figurative meaning) or LITERALLY.

Respond in EXACTLY this format, nothing else:
Label: [idiomatic/literal]

- Label must be exactly "idiomatic" or "literal"
- Do not add any explanation or extra text"""

LABEL2ID = {'literal': 0, 'idiomatic': 1}

def classify(sentence, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': f'Classify this sentence:\n\n"{sentence}"'},
                ],
                temperature=0,
                max_tokens=10,
            )
            raw = response.choices[0].message.content.strip().lower()
            for line in raw.split('\n'):
                if line.startswith('label:'):
                    val = line[6:].strip()
                    if 'idiomatic' in val: return 'idiomatic'
                    if 'literal'   in val: return 'literal'
            if 'literal'   in raw: return 'literal'
            if 'idiomatic' in raw: return 'idiomatic'
            return None
        except Exception as e:
            if attempt < retries - 1:
                print(f'  API error: {e} — retrying in 5s...')
                time.sleep(5)
            else:
                print(f'  API error after {retries} attempts: {e}')
                return None

def compute_metrics(preds):
    valid  = [p for p in preds if p['pred_label'] is not None]
    gold   = [LABEL2ID[p['idiomaticity']] for p in valid]
    pred   = [LABEL2ID[p['pred_label']]   for p in valid]
    macro  = f1_score(gold, pred, average='macro', zero_division=0)
    rep    = classification_report(gold, pred, target_names=['literal','idiomatic'],
                                   output_dict=True, zero_division=0)
    return {
        'macro_f1':     round(macro, 4),
        'literal_f1':   round(rep['literal']['f1-score'],   4),
        'idiomatic_f1': round(rep['idiomatic']['f1-score'], 4),
        'n':            len(valid),
        'n_parse_fail': len(preds) - len(valid),
    }

print('Functions defined')

Functions defined


In [6]:
# ── RUN ───────────────────────────────────────────────────────────────────────
gpt_results = {}
total_cost  = 0.0

for code, examples in lang_examples.items():
    if DRY_RUN:
        examples = examples[:DRY_RUN]
        print(f'[{code}] DRY RUN: {DRY_RUN} examples')

    preds_path = OUTPUT_DIR / f'id10m_gpt4o_{code.lower()}_preds.jsonl'

    # Resume: load already-done sentences
    completed = {}
    if not FORCE and preds_path.exists():
        for line in preds_path.open():
            r = json.loads(line)
            completed[r['sentence']] = r
        print(f'[{code}] Resuming: {len(completed)}/{len(examples)} already done')

    preds  = []
    n_fail = 0

    with preds_path.open('a' if completed else 'w') as f_out:
        for ex in tqdm(examples, desc=f'GPT-4o [{code}]'):
            sentence = ex['sentence']

            if sentence in completed:
                preds.append(completed[sentence])
                continue

            pred_label = classify(sentence)
            if pred_label is None:
                n_fail += 1
                pred_label = 'idiomatic'  # conservative default

            result = {
                'sentence':     sentence,
                'idiomaticity': ex['idiomaticity'],
                'pred_label':   pred_label,
                'correct':      pred_label == ex['idiomaticity'],
                'language':     ex.get('language', code),
                'id10m_id':     ex.get('id10m_id', ''),
            }
            preds.append(result)
            f_out.write(json.dumps(result, ensure_ascii=False) + '\n')
            f_out.flush()
            total_cost += (150 * 2.50 + 5 * 10.0) / 1_000_000
            time.sleep(SLEEP)

    # Verify write
    actual = sum(1 for _ in preds_path.open())
    assert actual == len(preds), f'Persistence FAILED: {actual} lines vs {len(preds)} preds'

    m = compute_metrics(preds)
    gpt_results[code] = m
    print(f'[{code}] n={m["n"]}  parse_fail={n_fail}')
    print(f'  GPT-4o macro F1 : {m["macro_f1"]:.4f}  (lit={m["literal_f1"]:.4f}, idiom={m["idiomatic_f1"]:.4f})')
    if code in mbert_results:
        mb = mbert_results[code]['macro_f1']
        print(f'  mBERT  macro F1 : {mb:.4f}  Δ={m["macro_f1"]-mb:+.4f}')

print(f'\nEstimated cost: ${total_cost:.3f}')

[EN] Resuming: 3/200 already done


GPT-4o [EN]:   0%|          | 0/200 [00:00<?, ?it/s]

[EN] n=200  parse_fail=0
  GPT-4o macro F1 : 0.8544  (lit=0.7606, idiom=0.9483)


GPT-4o [ES]:   0%|          | 0/199 [00:00<?, ?it/s]

[ES] n=199  parse_fail=0
  GPT-4o macro F1 : 0.6379  (lit=0.4468, idiom=0.8289)

Estimated cost: $0.168


In [8]:
# ── SAVE + COMPARISON TABLE ───────────────────────────────────────────────────
output = {
    'model':    MODEL,
    'approach': 'zero_shot_classification',
    'gpt4o':    gpt_results,
    'mbert':    mbert_results,
    'cost_est': round(total_cost, 4),
}
results_path = OUTPUT_DIR / 'id10m_gpt4o_results.json'
results_path.write_text(json.dumps(output, indent=2))
assert json.loads(results_path.read_text()) == output
print(f'Saved → {results_path}\n')

print('=' * 58)
print('  ID10M Zero-Shot: mBERT vs GPT-4o  |  CLS macro F1')
print('=' * 58)
print(f'  {"Lang":<8} {"mBERT":>10} {"GPT-4o":>10} {"Delta":>10} {"Winner":>10}')
print(f'  {"-"*8} {"-"*10} {"-"*10} {"-"*10} {"-"*10}')
for code in LANGS:
    if code not in gpt_results:
        continue
    g    = gpt_results[code]['macro_f1']
    m_val = mbert_results.get(code, {}).get('macro_f1', None)
    if m_val is not None:
        delta  = g - m_val
        winner = 'mBERT' if delta < 0 else 'GPT-4o'
        print(f'  {code:<8} {m_val:>10.4f} {g:>10.4f} {delta:>+10.4f} {winner:>10}')
    else:
        print(f'  {code:<8} {"N/A":>10} {g:>10.4f} {"N/A":>10} {"N/A":>10}')
print('=' * 58)

Saved → ID10M_test_eval_Output/id10m_gpt4o_results.json

  ID10M Zero-Shot: mBERT vs GPT-4o  |  CLS macro F1
  Lang          mBERT     GPT-4o      Delta     Winner
  -------- ---------- ---------- ---------- ----------
  EN              N/A     0.8544        N/A        N/A
  ES              N/A     0.6379        N/A        N/A
